# Lab 2.1: KV Caching Implementation (Enhanced)

## Objective
Implement and compare different Key-Value (KV) caching strategies for transformer inference. Understand memory-performance trade-offs and implement modern caching techniques.

## Learning Goals
1. Implement standard KV caching from scratch
2. Implement PagedAttention-style memory management
3. Explore advanced caching strategies (sliding window, GQA)
4. Profile cache performance and identify bottlenecks
5. Visualize memory usage and throughput trade-offs

## Prerequisites
- Completion of Week 1 labs
- Understanding of transformer attention mechanism
- Basic PyTorch and CUDA knowledge

## Modern Context
KV caching is critical for efficient LLM inference. Recent advances like vLLM's PagedAttention, FlashAttention-2, and Grouped-Query Attention have transformed caching strategies. This lab will guide you through implementing these techniques.

## Let's Begin!

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from typing import Optional, Tuple, Dict, List
import time
import psutil
import os
from dataclasses import dataclass
from collections import defaultdict
import matplotlib.pyplot as plt
import itertools

plt.style.use('seaborn-v0_8')
%matplotlib inline

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

## Part 1: Standard KV Caching Implementation

In this section, you'll implement a standard KV cache with pre-allocated memory. The cache stores key and value tensors for each layer, batch, head, and token position.

### Configuration
First, we define a configuration dataclass that holds cache parameters.

In [ ]:
@dataclass
class KVCacheConfig:
    """Configuration for KV caching."""
    max_batch_size: int = 4
    max_seq_len: int = 2048
    num_layers: int = 12
    num_heads: int = 12
    head_dim: int = 64
    dtype: torch.dtype = torch.float16
    device: torch.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    @property
    def hidden_size(self) -> int:
        return self.num_heads * self.head_dim
    
    @property
    def cache_shape_per_layer(self) -> Tuple[int, int, int, int]:
        """Shape: (batch_size, num_heads, seq_len, head_dim)"""
        return (self.max_batch_size, self.num_heads, self.max_seq_len, self.head_dim)
    
    @property
    def cache_size_per_layer_bytes(self) -> int:
        """Calculate cache size in bytes for one layer."""
        cache_elements = 2 * np.prod(self.cache_shape_per_layer)  # 2 for K and V
        bytes_per_element = torch.tensor([], dtype=self.dtype).element_size()
        return cache_elements * bytes_per_element
    
    @property
    def total_cache_size_bytes(self) -> int:
        """Total cache size for all layers."""
        return self.num_layers * self.cache_size_per_layer_bytes
    
    def print_memory_stats(self):
        """Print memory statistics for this configuration."""
        print(f"KV Cache Configuration:")
        print(f"  Max batch size: {self.max_batch_size}")
        print(f"  Max sequence length: {self.max_seq_len}")
        print(f"  Number of layers: {self.num_layers}")
        print(f"  Number of heads: {self.num_heads}")
        print(f"  Head dimension: {self.head_dim}")
        print(f"  Hidden size: {self.hidden_size}")
        print(f"  Data type: {self.dtype}")
        print(f"  Device: {self.device}")
        print(f"\nMemory Requirements:")
        print(f"  Cache per layer: {self.cache_size_per_layer_bytes / 1024**2:.2f} MB")
        print(f"  Total cache: {self.total_cache_size_bytes / 1024**2:.2f} MB")

# Let's create a config and examine memory usage
config = KVCacheConfig()
config.print_memory_stats()

### StandardKVCache Class

Now implement the `StandardKVCache` class. You need to complete the `update` and `get` methods.

The cache should:
1. Pre-allocate memory for all layers (keys and values)
2. Track current sequence length per batch
3. Update cache with new key-value pairs at the correct positions
4. Retrieve cached values for current sequence lengths

**Implementation hints:**
- Use `self.cache` dictionary mapping layer index to tuple (k_cache, v_cache)
- `self.current_seq_lens` tensor tracks lengths per batch
- In `update`, you need to copy `new_k` and `new_v` into the cache at positions `pos_start:pos_end`
- In `get`, slice the cache up to current sequence length for each batch index

Fill in the TODOs below.

In [ ]:
class StandardKVCache:
    """Standard KV cache implementation with pre-allocated memory."""
    
    def __init__(self, config: KVCacheConfig):
        self.config = config
        self.cache = {}  # layer_idx -> (k_cache, v_cache)
        self.current_seq_lens = torch.zeros(config.max_batch_size, dtype=torch.long, device=config.device)
        self._initialize_cache()
        
    def _initialize_cache(self):
        """Pre-allocate cache memory for all layers."""
        shape = self.config.cache_shape_per_layer
        
        for layer_idx in range(self.config.num_layers):
            # Initialize with zeros
            k_cache = torch.zeros(shape, dtype=self.config.dtype, device=self.config.device)
            v_cache = torch.zeros(shape, dtype=self.config.dtype, device=self.config.device)
            self.cache[layer_idx] = (k_cache, v_cache)
            
    def update(self, 
               layer_idx: int, 
               batch_indices: torch.Tensor, 
               new_k: torch.Tensor, 
               new_v: torch.Tensor, 
               positions: torch.Tensor) -> None:
        """
        Update cache with new key-value pairs.
        
        Args:
            layer_idx: Layer index
            batch_indices: [batch_size] indices for batch dimension
            new_k: [batch_size, num_heads, new_tokens, head_dim]
            new_v: [batch_size, num_heads, new_tokens, head_dim]
            positions: [batch_size, new_tokens] positions to store
        """
        # TODO: Implement cache update
        # 1. Get k_cache, v_cache for this layer
        # 2. For each batch index, update the cache at the appropriate positions
        # 3. Update self.current_seq_lens for each batch index
        # Get k_cache, v_cache for this layer
        k_cache, v_cache = self.cache[layer_idx]
        batch_size = batch_indices.shape[0]
        new_tokens = new_k.shape[2]
        
        # Update cache for each batch index
        for i in range(batch_size):
            batch_idx = batch_indices[i].item()
            # Scatter new tokens into cache at specified positions
            # positions[i] shape: [new_tokens]
            k_cache[batch_idx, :, positions[i], :] = new_k[i]
            v_cache[batch_idx, :, positions[i], :] = new_v[i]
        
        # Update current sequence lengths: max position + 1 for each batch
        for i in range(batch_size):
            batch_idx = batch_indices[i].item()
            max_pos = positions[i].max().item() + 1 if new_tokens > 0 else self.current_seq_lens[batch_idx]
            self.current_seq_lens[batch_idx] = max(max_pos, self.current_seq_lens[batch_idx])
        
    def get(self, layer_idx: int, batch_indices: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Get cached key-value pairs for current sequence lengths.
        
        Args:
            layer_idx: Layer index
            batch_indices: [batch_size] indices for batch dimension
            
        Returns:
            k_slice: [batch_size, num_heads, current_seq_len, head_dim]
            v_slice: [batch_size, num_heads, current_seq_len, head_dim]
        """
        # TODO: Implement cache retrieval
        # 1. Get k_cache, v_cache for this layer
        # 2. For each batch index, slice up to current_seq_lens[batch_idx]
        # 3. Stack slices along batch dimension
        # Get k_cache, v_cache for this layer
        k_cache, v_cache = self.cache[layer_idx]
        batch_size = batch_indices.shape[0]
        
        # Collect slices for each batch index
        k_slices = []
        v_slices = []
        for i in range(batch_size):
            batch_idx = batch_indices[i].item()
            seq_len = self.current_seq_lens[batch_idx]
            if seq_len > 0:
                k_slice = k_cache[batch_idx, :, :seq_len, :]
                v_slice = v_cache[batch_idx, :, :seq_len, :]
            else:
                # Return empty tensors with correct dimensions
                k_slice = torch.empty(
                    self.config.num_heads, 0, self.config.head_dim,
                    dtype=self.config.dtype, device=self.config.device
                )
                v_slice = torch.empty(
                    self.config.num_heads, 0, self.config.head_dim,
                    dtype=self.config.dtype, device=self.config.device
                )
            k_slices.append(k_slice)
            v_slices.append(v_slice)
        
        # Stack slices along batch dimension
        k_result = torch.stack(k_slices, dim=0)
        v_result = torch.stack(v_slices, dim=0)
        return k_result, v_result
    
    def reset_batch(self, batch_idx: int):
        """Reset cache for a specific batch index."""
        self.current_seq_lens[batch_idx] = 0
        
    def get_memory_usage(self) -> Dict[str, float]:
        """Get current memory usage in MB."""
        total_elements = 0
        bytes_per_element = torch.tensor([], dtype=self.config.dtype).element_size()
        
        for k_cache, v_cache in self.cache.values():
            total_elements += k_cache.numel() + v_cache.numel()
            
        allocated_mb = total_elements * bytes_per_element / 1024**2
        
        # Calculate actually used memory based on sequence lengths
        used_elements = 0
        max_seq_len = self.config.max_seq_len
        
        for batch_idx in range(self.config.max_batch_size):
            seq_len = self.current_seq_lens[batch_idx]
            used_elements += 2 * self.config.num_layers * self.config.num_heads * seq_len * self.config.head_dim
            
        used_mb = used_elements * bytes_per_element / 1024**2
        
        return {
            "allocated_mb": allocated_mb,
            "used_mb": used_mb,
            "utilization": used_mb / allocated_mb if allocated_mb > 0 else 0
        }

# Let's test your implementation
test_config = KVCacheConfig(max_batch_size=2, max_seq_len=512, num_layers=1, num_heads=4, head_dim=64)
cache = StandardKVCache(test_config)
print("Cache initialized.")

# Create dummy data
batch_indices = torch.tensor([0, 1], device=test_config.device)
new_k = torch.randn(2, 4, 10, 64, dtype=test_config.dtype, device=test_config.device)
new_v = torch.randn(2, 4, 10, 64, dtype=test_config.dtype, device=test_config.device)
positions = torch.tensor([[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]] * 2, device=test_config.device)

# TODO: Uncomment after implementing update
cache.update(0, batch_indices, new_k, new_v, positions)
print("Update completed.")

# TODO: Uncomment after implementing get
k, v = cache.get(0, batch_indices)
print(f"Retrieved shapes: k={k.shape}, v={v.shape}")
# print(f"Memory usage: {cache.get_memory_usage()}")

## Part 2: PagedAttention-style Cache Implementation

PagedAttention (from vLLM) divides the KV cache into fixed-size pages, similar to virtual memory. This allows more flexible memory allocation and reduces fragmentation.

### Page Configuration

In [ ]:
@dataclass
class PageConfig:
    """Configuration for paged cache."""
    page_size: int = 16  # Tokens per page
    num_pages: int = 128  # Maximum pages per block
    block_size: int = 16  # Pages per block

### PagedKVCache Class

Implement the `PagedKVCache` class. You need to complete the `update`, `get`, `allocate_page`, and `free_page` methods.

The cache should:
1. Pre-allocate a pool of pages (each page stores keys/values for a fixed number of tokens)
2. Maintain page tables mapping (batch, layer) -> list of page indices
3. Allocate new pages when needed, free pages when sequences are removed
4. Handle partial page filling

**Implementation hints:**
- `self.k_pages` and `self.v_pages` are tensors of shape `[num_pages, num_heads, page_size, head_dim]`
- `self.page_tables[batch_idx][layer_idx]` is a list of page indices
- `self.seq_lens` tracks total tokens per batch (not per layer)
- In `update`, you may need to allocate new pages if current pages are full
- In `get`, gather pages and concatenate tokens

Fill in the TODOs below.

In [ ]:
class PagedKVCache:
    """PagedAttention-style KV cache with dynamic memory allocation."""
    
    def __init__(self, config: KVCacheConfig, page_config: PageConfig):
        self.config = config
        self.page_config = page_config
        
        # Data structures for paging
        self.page_table = {}  # (batch_idx, layer_idx) -> list of page indices
        self.free_pages = list(range(page_config.num_pages))
        
        # Pre-allocate page memory
        page_shape = (config.num_heads, page_config.page_size, config.head_dim)
        self.k_pages = torch.zeros(
            page_config.num_pages, *page_shape, 
            dtype=config.dtype, device=config.device
        )
        self.v_pages = torch.zeros(
            page_config.num_pages, *page_shape,
            dtype=config.dtype, device=config.device
        )
        
        # Track sequence lengths
        self.seq_lens = torch.zeros(config.max_batch_size, dtype=torch.long, device=config.device)
        self.page_tables = {}
        
        self._initialize_page_tables()
        
    def _initialize_page_tables(self):
        """Initialize page tables for all batches and layers."""
        for batch_idx in range(self.config.max_batch_size):
            self.page_tables[batch_idx] = {}
            for layer_idx in range(self.config.num_layers):
                self.page_tables[batch_idx][layer_idx] = []
                
    def allocate_page(self) -> int:
        """Allocate a free page."""
        # TODO: Return a page index from free_pages, or raise error if none available
        raise NotImplementedError("Implement allocate_page")
    
    def free_page(self, page_idx: int):
        """Free a page for reuse."""
        # TODO: Add page_idx back to free_pages
        raise NotImplementedError("Implement free_page")
        
    def update(self, 
               layer_idx: int, 
               batch_idx: int, 
               new_k: torch.Tensor, 
               new_v: torch.Tensor) -> None:
        """
        Update cache with new key-value pairs using paging.
        
        Args:
            layer_idx: Layer index
            batch_idx: Batch index
            new_k: [num_heads, new_tokens, head_dim]
            new_v: [num_heads, new_tokens, head_dim]
        """
        # TODO: Implement paged update
        # 1. Determine how many tokens need to be stored
        # 2. Check if existing pages have space
        # 3. Allocate new pages if needed
        # 4. Copy tokens into appropriate pages
        # 5. Update seq_lens and page_tables
        # Get k_cache, v_cache for this layer
        k_cache, v_cache = self.cache[layer_idx]
        batch_size = batch_indices.shape[0]
        new_tokens = new_k.shape[2]
        
        # Update cache for each batch index
        for i in range(batch_size):
            batch_idx = batch_indices[i].item()
            # Scatter new tokens into cache at specified positions
            # positions[i] shape: [new_tokens]
            k_cache[batch_idx, :, positions[i], :] = new_k[i]
            v_cache[batch_idx, :, positions[i], :] = new_v[i]
        
        # Update current sequence lengths: max position + 1 for each batch
        for i in range(batch_size):
            batch_idx = batch_indices[i].item()
            max_pos = positions[i].max().item() + 1 if new_tokens > 0 else self.current_seq_lens[batch_idx]
            self.current_seq_lens[batch_idx] = max(max_pos, self.current_seq_lens[batch_idx])
    
    def get(self, layer_idx: int, batch_idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Get cached key-value pairs for current sequence length.
        
        Args:
            layer_idx: Layer index
            batch_idx: Batch index
            
        Returns:
            k: [num_heads, seq_len, head_dim]
            v: [num_heads, seq_len, head_dim]
        """
        # TODO: Implement paged retrieval
        # 1. Get page indices for this batch and layer
        # 2. Gather pages and concatenate tokens
        # 3. Return sliced up to current seq_len
        raise NotImplementedError("Implement get method")
        
    def reset_batch(self, batch_idx: int):
        """Reset cache for a specific batch index."""
        # Free all pages used by this batch across all layers
        for layer_idx in range(self.config.num_layers):
            for page_idx in self.page_tables[batch_idx][layer_idx]:
                self.free_page(page_idx)
            self.page_tables[batch_idx][layer_idx] = []
        self.seq_lens[batch_idx] = 0
    
    def get_memory_usage(self) -> Dict[str, float]:
        """Get current memory usage in MB."""
        bytes_per_element = torch.tensor([], dtype=self.config.dtype).element_size()
        allocated_elements = self.k_pages.numel() + self.v_pages.numel()
        allocated_mb = allocated_elements * bytes_per_element / 1024**2
        
        # Calculate used memory based on allocated pages
        used_pages = self.page_config.num_pages - len(self.free_pages)
        used_elements = used_pages * self.config.num_heads * self.page_config.page_size * self.config.head_dim * 2
        used_mb = used_elements * bytes_per_element / 1024**2
        
        return {
            "allocated_mb": allocated_mb,
            "used_mb": used_mb,
            "utilization": used_mb / allocated_mb if allocated_mb > 0 else 0
        }

# Test your implementation
page_config = PageConfig(page_size=16, num_pages=32)
paged_cache = PagedKVCache(test_config, page_config)
print("Paged cache initialized.")

# TODO: Create test data and test update/get methods
# new_k = torch.randn(4, 20, 64, dtype=test_config.dtype, device=test_config.device)
# new_v = torch.randn(4, 20, 64, dtype=test_config.dtype, device=test_config.device)
# paged_cache.update(0, 0, new_k, new_v)
# k, v = paged_cache.get(0, 0)
print(f"Retrieved shapes: k={k.shape}, v={v.shape}")
# print(f"Memory usage: {paged_cache.get_memory_usage()}")

## Part 3: Advanced Caching Strategies

Now that you have implemented basic caching, let's explore advanced strategies.

### Sliding Window Cache
Limit cache size to a fixed window of recent tokens. Old tokens are evicted when window is full.

### Grouped-Query Attention (GQA)
Reduce cache size by grouping query heads that share key/value heads.

### Your Task
Choose one advanced strategy to implement. You can extend either `StandardKVCache` or `PagedKVCache`.

In [ ]:
# TODO: Implement one advanced caching strategy

class SlidingWindowCache:
    """KV cache with sliding window eviction."""
    pass

class GQACache:
    """KV cache for Grouped-Query Attention."""
    pass

# Or modify existing classes to support these features

## Part 4: Benchmarking and Visualization

Now compare the performance of your implementations.

We'll measure:
1. Memory usage vs sequence length
2. Update/retrieval latency
3. Cache utilization

Fill in the benchmarking code below.

In [ ]:
def benchmark_cache(cache_constructor, config, page_config=None, max_tokens=1024, step=64):
    """Benchmark cache performance."""
    results = []
    
    for seq_len in range(step, max_tokens + 1, step):
        # Create cache
        if page_config:
            cache = cache_constructor(config, page_config)
        else:
            cache = cache_constructor(config)
        
        # Measure update time
        start = time.time()
        # TODO: Simulate sequential token addition
        # For each token, update cache with dummy data
        end = time.time()
        update_time = end - start
        
        # Measure retrieval time
        start = time.time()
        # TODO: Retrieve cached values
        end = time.time()
        retrieve_time = end - start
        
        # Get memory usage
        mem = cache.get_memory_usage()
        
        results.append({
            "seq_len": seq_len,
            "update_time": update_time,
            "retrieve_time": retrieve_time,
            "memory_mb": mem["used_mb"],
            "utilization": mem["utilization"]
        })
    
    return results

# TODO: Run benchmarks for both cache types
# standard_results = benchmark_cache(StandardKVCache, test_config, max_tokens=512)
# paged_results = benchmark_cache(PagedKVCache, test_config, page_config, max_tokens=512)

# TODO: Plot results
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
# Your plotting code here
plt.tight_layout()
plt.show()

## Conclusion

In this lab, you implemented:
1. Standard KV caching with pre-allocated memory
2. PagedAttention-style caching with dynamic allocation
3. (Optional) Advanced caching strategy

### Key Takeaways
- **Standard caching**: Simple but memory-inefficient for variable-length sequences
- **Paged caching**: More complex but enables memory sharing and reduces fragmentation
- **Trade-offs**: Different strategies suit different workloads

### Further Exploration
1. Integrate your cache with an actual transformer model
2. Compare with vLLM's implementation
3. Experiment with different page sizes and eviction policies
4. Implement FlashAttention-2 with your caching system

### Submission
Submit your completed notebook with all implementations and benchmarking results.